# Feature Extraction from Pretrained Models

Este notebook permite extrair features de modelos pré-treinados e salvar em pastas customizadas.

In [ ]:
import sys
sys.path.insert(0, '/home/eflammere/HybridEmbeddingsForBreastCancerDetection')

import os
import numpy as np
import torch
from tqdm import tqdm

from utils.setup import setup_device
from src.utils.dataloader import BreastMNISTLoader

# Importar TODOS os modelos quânticos
from src.models.main.models import (
    QuantumModel,                      # Quantum sem encoder (non-trainable)
    QuantumModelWithEncoder,           # Quantum com encoder (non-trainable)
    ExtentedQuantumModel,              # Quantum sem encoder (trainable, 3 FC layers)
    ExtentedQuantumModelWithEncoder,   # Quantum com encoder (trainable, 3 FC layers)
)

# Importar TODOS os modelos clássicos
from src.models.main.classical import (
    ClassicalModel,                    # ResNet18 sem encoder
    ClassicalModelWithEncoder,         # ResNet18 com encoder
    ShallowCNN,                        # CNN rasa sem encoder
    ShallowCNNWithEncoder,             # CNN rasa com encoder
)

# Importar TODOS os extractors disponíveis
from models.extraction import (
    ClassicalFeatureExtractor,                # Para modelos clássicos SEM encoder
    ClassicalFeatureExtractorWithEncoder,     # Para modelos clássicos COM encoder
    QuantumFeatureExtractor,                  # Para modelos quânticos SEM encoder
    QuantumFeatureExtractorWithEncoder,       # Para modelos quânticos COM encoder
    GenericFeatureExtractor,                  # Detecção automática
    create_extractor,                         # Helper para criar extractor automaticamente
)

## 1. Configurações

Configure os caminhos dos modelos e a pasta de saída:

In [11]:
# Caminhos dos modelos pré-treinados
QUANTUM_MODEL_PATH = "experiments/models/quantum/trainable/BUSUCLM/checkpoints/model_epoch5.pth"
CLASSICAL_MODEL_PATH = "experiments/models/classical/BUSUCLM/resnet/best_resnet.pth"

# Pasta de saída (você pode mudar para qualquer nome)
OUTPUT_DIR = "experiments/features"

# Parâmetros dos modelos
N_QUBITS = 4
N_LAYERS = 1
NUM_CLASSES = 2

# Parâmetros do dataloader
BATCH_SIZE = 8
IMAGE_SIZE = 28

## 2. Carregar Modelos Pré-treinados

In [12]:
# Setup device
device = setup_device()
print(f"Using device: {device}")

# ============================================================================
# FUNÇÃO PARA DETECTAR TIPO DE MODELO QUÂNTICO AUTOMATICAMENTE
# ============================================================================

def detect_quantum_model_type(checkpoint_path):
    """Detecta se o checkpoint é de QuantumModel ou ExtentedQuantumModel."""
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    
    # Extrair state_dict
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "state_dict", "model"):
            if key in checkpoint:
                checkpoint = checkpoint[key]
                break
    
    # Verificar chaves
    keys = set(checkpoint.keys())
    print(f"  ℹ️ Primeiras chaves do checkpoint: {list(keys)[:5]}")
    
    # ExtentedQuantumModel tem quanv.* (trainable quantum layer)
    if any('quanv.' in k for k in keys):
        return "ExtentedQuantumModel"
    # QuantumModel básico tem apenas fc (e não tem quanv)
    elif 'fc.weight' in keys and not any('quanv.' in k for k in keys):
        return "QuantumModel"
    else:
        print(f"⚠️ Não foi possível detectar o tipo. Chaves: {list(keys)[:10]}")
        return None

# ============================================================================
# DETECTAR E CRIAR MODELO QUÂNTICO APROPRIADO
# ============================================================================

print("\nDetectando tipo de modelo quântico...")
quantum_type = detect_quantum_model_type(QUANTUM_MODEL_PATH)
print(f"  → Detectado: {quantum_type}")

if quantum_type == "ExtentedQuantumModel":
    quantum_model = ExtentedQuantumModel(
        n_qubits=N_QUBITS, 
        n_layers=N_LAYERS,
        num_classes=NUM_CLASSES, 
        patch_size=2,
        mode="2d"
    )
    print("  ✓ Criado ExtentedQuantumModel (trainable)")
elif quantum_type == "QuantumModel":
    # Gerar parâmetros aleatórios com a forma correta (n_layers, n_qubits)
    import numpy as np
    rand_params = np.random.uniform(low=0, high=2*np.pi, size=(N_LAYERS, N_QUBITS))
    print(f"  ℹ️ Parâmetros aleatórios gerados: shape={rand_params.shape}")
    
    quantum_model = QuantumModel(
        n_qubits=N_QUBITS, 
        n_layers=N_LAYERS, 
        rand_params=rand_params,  # Array numpy, não booleano
        num_classes=NUM_CLASSES,
        patch_size=2
    )
    print("  ✓ Criado QuantumModel (non-trainable)")
else:
    raise ValueError("❌ Não foi possível detectar o tipo de modelo quântico!")

# Criar modelo clássico
classical_model = ShallowCNN(num_classes=NUM_CLASSES)

# ============================================================================
# CARREGAR PESOS PRÉ-TREINADOS
# ============================================================================

print("\nCarregando modelos...")

def _load_model_checkpoint(model, checkpoint_path, device):
	"""Carrega checkpoint de modelo, lidando com diferentes formatos."""
	checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
	
	# Extrair state_dict se estiver em um dicionário wrapper
	if isinstance(checkpoint, dict):
		for key in ("model_state_dict", "state_dict", "model"):
			if key in checkpoint:
				checkpoint = checkpoint[key]
				break
	
	# Remover prefixo 'module.' se existir (de DataParallel)
	if isinstance(checkpoint, dict):
		checkpoint = {
			(k.replace("module.", "", 1) if k.startswith("module.") else k): v
			for k, v in checkpoint.items()
		}
	
	# Para modelos quânticos trainable (ExtendedQuantumModel), inicializar primeiro
	if hasattr(model, 'quanv'):
		print(f"  ℹ️ Inicializando modelo quântico trainable...")
		model.to(device)
		with torch.no_grad():
			dummy_input = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device)
			try:
				_ = model(dummy_input)
				print(f"  ✓ Inicialização bem-sucedida")
			except Exception as e:
				print(f"  ⚠️ Erro na inicialização: {e}")
				raise
		
		# Carregar APENAS os pesos da camada FC (não sobrescrever circuitos quânticos)
		print(f"  ℹ️ Carregando pesos do checkpoint (apenas FC)...")
		fc_state = {k: v for k, v in checkpoint.items() if k.startswith('fc.')}
		if fc_state:
			model.load_state_dict(fc_state, strict=False)
			print(f"  ✓ Carregados pesos FC: {list(fc_state.keys())}")
		else:
			print(f"  ⚠️ Nenhum peso FC encontrado no checkpoint")
		return model
	
	# Para modelos com camadas dinâmicas (fc=None), inicializar com dummy input
	if hasattr(model, 'fc') and model.fc is None:
		print(f"  ℹ️ Inicializando camadas dinâmicas do {type(model).__name__}...")
		model.to(device)
		with torch.no_grad():
			dummy_input = torch.randn(1, 1, IMAGE_SIZE, IMAGE_SIZE).to(device)
			try:
				_ = model(dummy_input)
			except Exception as e:
				print(f"  ⚠️ Aviso ao inicializar: {e}")
	
	# Carregar state_dict completo
	missing_keys, unexpected_keys = model.load_state_dict(checkpoint, strict=False)
	
	if missing_keys:
		print(f"  ⚠️ Chaves faltantes: {missing_keys}")
	if unexpected_keys:
		print(f"  ⚠️ Chaves inesperadas (ignoradas): {unexpected_keys}")
	
	return model

# Carregar ambos os modelos
print("  → Carregando quantum model...")
_load_model_checkpoint(quantum_model, QUANTUM_MODEL_PATH, device)
print("  → Carregando classical model...")
_load_model_checkpoint(classical_model, CLASSICAL_MODEL_PATH, device)

# Mover para device e modo eval
quantum_model.to(device).eval()
classical_model.to(device).eval()

print("\n✓ Modelos carregados com sucesso!")
print(f"  - Quantum: {type(quantum_model).__name__}")
print(f"  - Classical: {type(classical_model).__name__}")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4
Using device: cuda

Detectando tipo de modelo quântico...
  ℹ️ Primeiras chaves do checkpoint: ['fc.bias', 'quanv.qlayer.weights', 'fc.weight']
  → Detectado: ExtentedQuantumModel
Quantum layer has 4 trainable parameters
Requires grad: True
  ✓ Criado ExtentedQuantumModel (trainable)

Carregando modelos...
  → Carregando quantum model...
  ℹ️ Inicializando modelo quântico trainable...
  ✓ Inicialização bem-sucedida
  ℹ️ Carregando pesos do checkpoint (apenas FC)...
  ✓ Carregados pesos FC: ['fc.weight', 'fc.bias']
  → Carregando classical model...
  ⚠️ Chaves faltantes: ['conv1.0.weight', 'conv1.0.bias', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv2.0.weight', 'conv2.0.bias', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv3.0.weight', 'conv3.0.bias', 'conv3.1.weight', 'conv3.1.bias', 'conv3.1.running_mean', 'conv3.1.running_var', 'fc.weight', 'fc.bias']
  

## 3. Criar Feature Extractors

## 3. Escolher Modo de Extração

Existem diferentes modos de extração dependendo do tipo de modelo:

### Modelos Disponíveis e seus Extractors:

#### **Modelos Clássicos:**
| Modelo | Encoder? | Extractor |
|--------|----------|-----------|
| `ClassicalModel` (ResNet18) | ❌ | `ClassicalFeatureExtractor` |
| `ClassicalModelWithEncoder` | ✅ | `ClassicalFeatureExtractorWithEncoder` |
| `ShallowCNN` | ❌ | `ClassicalFeatureExtractor` |
| `ShallowCNNWithEncoder` | ✅ | `ClassicalFeatureExtractorWithEncoder` |

#### **Modelos Quânticos:**
| Modelo | Encoder? | Tipo | Extractor |
|--------|----------|------|-----------|
| `QuantumModel` | ❌ | Non-trainable | `QuantumFeatureExtractor` |
| `QuantumModelWithEncoder` | ✅ | Non-trainable | `QuantumFeatureExtractorWithEncoder` |
| `ExtentedQuantumModel` | ❌ | Trainable (3 FC) | `QuantumFeatureExtractor` |
| `ExtentedQuantumModelWithEncoder` | ✅ | Trainable (3 FC) | `QuantumFeatureExtractorWithEncoder` |
| `DualStreamQuantumModel` | ✅ | Dual-stream | `QuantumFeatureExtractorWithEncoder` |

### Opções de Extração:

1. **Manual**: Escolha o extractor específico baseado na tabela acima
2. **Automática**: Use `create_extractor()` ou `GenericFeatureExtractor` - detecta automaticamente o tipo

**Descomente a opção apropriada abaixo:**

In [13]:
# ============================================================================
# ESCOLHA O MODO DE EXTRAÇÃO APROPRIADO PARA SEUS MODELOS
# ============================================================================

# OPÇÃO 1: Modelos SEM encoder (padrão - ResNet18, ShallowCNN, QuantumModel básico)
# Descomente estas linhas se seus modelos NÃO usam encoder:
classical_extractor = ClassicalFeatureExtractor(classical_model).to(device)
quantum_extractor = QuantumFeatureExtractor(quantum_model).to(device)

# OPÇÃO 2: Modelos COM encoder
# Descomente estas linhas se seus modelos USAM encoder:
# classical_extractor = ClassicalFeatureExtractorWithEncoder(classical_model).to(device)
# quantum_extractor = QuantumFeatureExtractorWithEncoder(quantum_model).to(device)

# OPÇÃO 3: Detecção automática (recomendado se não tem certeza)
# Descomente estas linhas para detectar automaticamente:
# classical_extractor = create_extractor(classical_model).to(device)
# quantum_extractor = create_extractor(quantum_model).to(device)

# OPÇÃO 4: Usar GenericFeatureExtractor diretamente
# Descomente estas linhas:
# classical_extractor = GenericFeatureExtractor(classical_model).to(device)
# quantum_extractor = GenericFeatureExtractor(quantum_model).to(device)

# ============================================================================

print("✓ Feature extractors criados!")
print(f"  - Classical extractor: {type(classical_extractor).__name__}")
print(f"  - Quantum extractor: {type(quantum_extractor).__name__}")

✓ Feature extractors criados!
  - Classical extractor: ClassicalFeatureExtractor
  - Quantum extractor: QuantumFeatureExtractor


## 4. Preparar Dados

In [14]:
# Carregar dataloaders
loader = BreastMNISTLoader(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE)
train_loader, test_loader, val_loader = loader.get_dataloaders()

splits = {
    "train": train_loader,
    "test": test_loader,
    "val": val_loader
}

print(f"✓ Dataloaders prontos:")
print(f"  - Train batches: {len(train_loader)}")
print(f"  - Test batches: {len(test_loader)}")
print(f"  - Val batches: {len(val_loader)}")

Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/breastmnist.npz
✓ Dataloaders prontos:
  - Train batches: 69
  - Test batches: 20
  - Val batches: 10


## 5. Função de Extração

Esta função extrai as features usando o extractor escolhido acima.

**Métodos de extração customizados:**
- Se quiser modificar o método de extração, edite a linha `features = extractor(imgs)`
- Exemplos de customização:
  - Extrair apenas da camada específica
  - Aplicar transformações adicionais
  - Combinar múltiplas camadas

In [15]:
def extract_features(extractor, dataloader, output_dir, split_name, target_dim=None):
    """Extrai features de um dataloader e salva.
    
    Args:
        extractor: Feature extractor
        dataloader: DataLoader
        output_dir: Diretório de saída
        split_name: Nome do split
        target_dim: Dimensão alvo (opcional). Se especificado, reduz features para essa dimensão
    """
    os.makedirs(output_dir, exist_ok=True)
    
    features_list = []
    labels_list = []
    
    extractor.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(dataloader, desc=f"Extraindo {split_name}"):
            imgs = imgs.to(device)
            
            # ================================================================
            # MÉTODO DE EXTRAÇÃO - Modifique aqui se necessário
            # ================================================================
            
            # Método padrão: usar o extractor diretamente
            features = extractor(imgs)
            
            # Se target_dim especificado, reduzir dimensionalidade
            if target_dim is not None and features.shape[1] != target_dim:
                # Reduzir usando adaptive pooling 1D
                features = features.view(features.size(0), -1)
                features = torch.nn.functional.adaptive_avg_pool1d(
                    features.unsqueeze(1), 
                    target_dim
                ).squeeze(1)
            
            # Exemplos de métodos customizados (comente o padrão e descomente um destes):
            
            # Exemplo 1: Normalizar features após extração
            # features = extractor(imgs)
            # features = torch.nn.functional.normalize(features, p=2, dim=1)
            
            # Exemplo 2: Aplicar pooling adicional manual
            # features = extractor(imgs)
            # features = features.view(features.size(0), -1)
            # features = torch.nn.functional.adaptive_avg_pool1d(features.unsqueeze(1), 128).squeeze(1)
            
            # Exemplo 3: Extrair e concatenar múltiplas camadas (requer modificação do extractor)
            # features_layer1 = extractor.layer1(imgs)
            # features_layer2 = extractor.layer2(imgs)
            # features = torch.cat([features_layer1, features_layer2], dim=1)
            
            # ================================================================
            
            features_list.append(features.cpu().numpy())
            labels_list.append(labels.cpu().numpy())
    
    # Concatenar e salvar
    features = np.concatenate(features_list, axis=0)
    labels = np.concatenate(labels_list, axis=0)
    
    np.save(os.path.join(output_dir, f"{split_name}_features.npy"), features)
    np.save(os.path.join(output_dir, f"{split_name}_labels.npy"), labels)
    
    print(f"  ✓ Salvos: {split_name}_features.npy e {split_name}_labels.npy")
    
    return features.shape

## 6. Extrair Features

Execute a extração para todos os splits:

In [ ]:
# Criar diretórios de saída
classical_dir = os.path.join(OUTPUT_DIR, "classical")
quantum_dir = os.path.join(OUTPUT_DIR, "quantum")

os.makedirs(classical_dir, exist_ok=True)
os.makedirs(quantum_dir, exist_ok=True)

print(f"Salvando features em: {OUTPUT_DIR}\n")

# ============================================================================
# CONFIGURAÇÃO: Dimensão alvo das features
# ============================================================================
# Defina a dimensão desejada para todas as features (None = manter original)
TARGET_FEATURE_DIM = 128  # Mude para None se quiser manter dimensões originais
# ============================================================================

# Extrair para cada split
for split_name, split_loader in splits.items():
    print(f"\n{'='*60}")
    print(f"Processando: {split_name.upper()}")
    print(f"{'='*60}")
    
    # Features clássicas
    print("\n[Classical]")
    c_shape = extract_features(classical_extractor, split_loader, classical_dir, 
                               split_name, target_dim=TARGET_FEATURE_DIM)
    print(f"  Shape: {c_shape}")
    
    # Features quânticas
    print("\n[Quantum]")
    q_shape = extract_features(quantum_extractor, split_loader, quantum_dir, 
                               split_name, target_dim=TARGET_FEATURE_DIM)
    print(f"  Shape: {q_shape}")

print(f"\n\n{'='*60}")
print("✓ EXTRAÇÃO COMPLETA!")
print(f"{'='*60}")
print(f"\nArquivos salvos em: {OUTPUT_DIR}")
if TARGET_FEATURE_DIM is not None:
    print(f"Dimensão das features: {TARGET_FEATURE_DIM}")

Salvando features em: experiments/features


Processando: TRAIN

[Classical]


Extraindo train: 100%|██████████| 69/69 [00:00<00:00, 391.73it/s]


  ✓ Salvos: train_features.npy e train_labels.npy
  Shape: (546, 128)

[Quantum]


Extraindo train: 100%|██████████| 69/69 [02:10<00:00,  1.89s/it]


  ✓ Salvos: train_features.npy e train_labels.npy
  Shape: (546, 128)

Processando: TEST

[Classical]


Extraindo test: 100%|██████████| 20/20 [00:00<00:00, 391.11it/s]


  ✓ Salvos: test_features.npy e test_labels.npy
  Shape: (156, 128)

[Quantum]


Extraindo test:  10%|█         | 2/20 [00:04<00:36,  2.02s/it]Exception ignored in: <function _xla_gc_callback at 0x7b24883db4c0>
Traceback (most recent call last):
  File "/home/eflammere/miniconda3/envs/quantumEnvironment/lib/python3.12/site-packages/jax/_src/lib/__init__.py", line 113, in _xla_gc_callback
    xla_client._xla.collect_garbage()
KeyboardInterrupt: 
Extraindo test:  40%|████      | 8/20 [00:15<00:23,  1.95s/it]

## 7. Verificar Features Extraídas

Verifique as dimensões e características das features extraídas:

In [ ]:
# Carregar e verificar features de todos os splits
print("="*60)
print("RESUMO DAS FEATURES EXTRAÍDAS")
print("="*60)

for split_name in ["train", "val", "test"]:
    print(f"\n{split_name.upper()} Split:")
    
    c_features = np.load(os.path.join(classical_dir, f"{split_name}_features.npy"))
    q_features = np.load(os.path.join(quantum_dir, f"{split_name}_features.npy"))
    labels = np.load(os.path.join(classical_dir, f"{split_name}_labels.npy"))
    
    print(f"  Classical features: {c_features.shape}")
    print(f"  Quantum features:   {q_features.shape}")
    print(f"  Labels:             {labels.shape}")

print(f"\n{'='*60}")
print("Arquivos salvos em:")
print(f"  Classical: {classical_dir}/")
print(f"  Quantum:   {quantum_dir}/")
print("="*60)

RESUMO DAS FEATURES EXTRAÍDAS

TRAIN Split:
  Classical features: (546, 128)
  Quantum features:   (546, 128)
  Labels:             (546, 1)

VAL Split:
  Classical features: (78, 128)
  Quantum features:   (78, 128)
  Labels:             (78, 1)

TEST Split:
  Classical features: (156, 128)
  Quantum features:   (156, 128)
  Labels:             (156, 1)

Arquivos salvos em:
  Classical: experiments/features/classical/
  Quantum:   experiments/features/quantum/


## 8. (Opcional) Usar a Função Helper

Alternativamente, você pode usar a função helper `extract_features_from_dataloader()` que faz tudo automaticamente:

In [ ]:
# Exemplo: extrair features de forma mais simples usando a função helper
from models.extraction import extract_features_from_dataloader

# Extrair de um split específico (exemplo: test)
# features, labels = extract_features_from_dataloader(classical_model, test_loader, device)
# print(f"Features extraídas: {features.shape}")
# print(f"Labels: {labels.shape}")

# Salvar
# np.save("test_features_alternative.npy", features)
# np.save("test_labels_alternative.npy", labels)

print("Função helper disponível: extract_features_from_dataloader()")
print("Descomente o código acima para usar.")

Função helper disponível: extract_features_from_dataloader()
Descomente o código acima para usar.
